In [0]:
%sql
DROP TABLE IF EXISTS workspace.raw.clientes;
DROP TABLE IF EXISTS workspace.raw.contas;
DROP TABLE IF EXISTS workspace.raw.formas_pagamento;
DROP TABLE IF EXISTS workspace.raw.municipios;
DROP TABLE IF EXISTS workspace.raw.estabelecimentos;
DROP TABLE IF EXISTS workspace.raw.transacoes;

In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/raw/landing"
CONTROLE = f"{LANDING}/_controle"          # schemas e checkpoints do Auto Loader

# Uma linha por fonte: pasta de origem, tabela de destino, formato e opções de leitura
OPCOES_CSV = {"header": "true", "sep": ";"}
FONTES = [
    {"origem": "crm/clientes",                  "tabela": "clientes",         "formato": "csv",     "opcoes": OPCOES_CSV},
    {"origem": "crm/contas",                    "tabela": "contas",           "formato": "csv",     "opcoes": OPCOES_CSV},
    {"origem": "referencia/formas_pagamento",   "tabela": "formas_pagamento", "formato": "csv",     "opcoes": OPCOES_CSV},
    {"origem": "referencia/municipios",         "tabela": "municipios",       "formato": "csv",     "opcoes": OPCOES_CSV},
    {"origem": "parceiros/estabelecimentos",    "tabela": "estabelecimentos", "formato": "parquet", "opcoes": {}},
    {"origem": "core/transacoes",               "tabela": "transacoes",       "formato": "json",    "opcoes": {}},
]


def ingerir_raw(fonte):
    tabela = f"workspace.raw.{fonte['tabela']}"
    leitura = (spark.readStream.format("cloudFiles")
               .option("cloudFiles.format", fonte["formato"])
               .option("cloudFiles.schemaLocation", f"{CONTROLE}/schemas/{fonte['tabela']}"))
    for chave, valor in fonte["opcoes"].items():
        leitura = leitura.option(chave, valor)

    (leitura.load(f"{LANDING}/{fonte['origem']}/")
        .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("_ingerido_em", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{CONTROLE}/checkpoints/{fonte['tabela']}")
        .trigger(availableNow=True)
        .toTable(tabela)
        .awaitTermination())

    print(f"{tabela}: {spark.table(tabela).count()} registros")


for fonte in FONTES:
    ingerir_raw(fonte)